# Crear y probar perfiles de voz

XTTS-v2 crea un perfil reutilizable a partir de cada grabación y sintetiza audio con el perfil elegido.

In [93]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


## Entorno y dispositivo

Se usa CUDA si está disponible; en CPU la ejecución tarda más. Dependencias: `requirements-lab4-modular.txt`.

In [94]:
import os
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    try:
        torch.arange(1, device='cuda')
    except RuntimeError as exc:
        print('CUDA no puede ejecutar kernels con este PyTorch; usaré CPU:', exc)
        DEVICE = 'cpu'
print('Dispositivo:', DEVICE)
print('CUDA disponible:', torch.cuda.is_available())


Dispositivo: cuda
CUDA disponible: True


In [95]:
from pathlib import Path

# Encontrar la raíz aunque Jupyter se haya iniciado desde notebooks/.
PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data" / "voice_samples").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró data/voice_samples. Inicia Jupyter desde la raíz del proyecto."
    )

VOICE_SAMPLES_DIR = PROJECT_DIR / "data" / "voice_samples"
VOICE_PROFILES_DIR = PROJECT_DIR / "data" / "voice_profiles"
VOICE_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
voice_sample_paths = sorted(VOICE_SAMPLES_DIR.glob("*.wav"))
if not voice_sample_paths:
    raise FileNotFoundError(f"No hay archivos WAV en {VOICE_SAMPLES_DIR.resolve()}")

available_voice_ids = {path.stem for path in voice_sample_paths}
import ipywidgets as widgets
from IPython.display import display

print("Raíz del proyecto:", PROJECT_DIR)
print(f"Audios encontrados: {len(voice_sample_paths)}")
for path in voice_sample_paths:
    print(" -", path.stem, "→", path.name)


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz
Audios encontrados: 6
 - ana → ana.wav
 - diego → diego.wav
 - elsa → elsa.wav
 - estuardo → estuardo.wav
 - nicole → nicole.wav
 - winz → winz.wav


In [96]:
# El selector de prueba está junto a la celda de síntesis.
print(f"Perfiles preparados: {len(list(VOICE_PROFILES_DIR.glob('*.pt')))}")

Perfiles preparados: 6


## Cargar XTTS-v2

El modelo multilingüe sintetiza audio y se descarga en la primera ejecución. La celda configura la carga compatible del checkpoint.

In [97]:
import os
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.api import TTS

# XTTS-v2 oficial serializa estas clases de configuración en su checkpoint.
torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig])
os.environ.setdefault("COQUI_TOS_AGREED", "1")

# Compatibilidad acotada a la carga del checkpoint XTTS oficial; luego se restaura.
_anterior_weights_only = os.environ.get("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
try:
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)
finally:
    if _anterior_weights_only is None:
        os.environ.pop("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", None)
    else:
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = _anterior_weights_only

model = tts.synthesizer.tts_model
print("XTTS-v2 cargado")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
XTTS-v2 cargado


## Grabaciones de referencia

Coloca un WAV autorizado por voz en `data/voice_samples/`. El nombre del archivo identifica el perfil, por ejemplo `winz.wav` genera `winz.pt`. Usa una grabación limpia de una sola persona, de 10 a 30 segundos.

## Crear perfiles

XTTS calcula `gpt_cond_latent` y `speaker_embedding` para cada WAV y guarda un archivo `.pt` por voz.

In [98]:
import torch
import torchaudio

print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("TorchAudio:", torchaudio.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capacidad GPU:", torch.cuda.get_device_capability(0))
    print("Arquitecturas de PyTorch:", torch.cuda.get_arch_list())

if DEVICE == "cuda":
    print("Prueba básica de CUDA: OK")
else:
    print("Se continúa con CPU.")

PyTorch: 2.7.1+cu128
CUDA de PyTorch: 12.8
TorchAudio: 2.7.1+cu128
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Capacidad GPU: (12, 0)
Arquitecturas de PyTorch: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120', 'compute_120']
Prueba básica de CUDA: OK


In [99]:
generated_profiles = []
failed_profiles = []

for voice_sample_path in voice_sample_paths:
    voice_id = voice_sample_path.stem
    profile_path = VOICE_PROFILES_DIR / f"{voice_id}.pt"
    try:
        with torch.inference_mode():
            gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
                audio_path=[str(voice_sample_path)],
                gpt_cond_len=6,
                max_ref_length=30,
                sound_norm_refs=False
            )
        torch.save({
            "voice_id": voice_id,
            "gpt_cond_latent": gpt_cond_latent.detach().cpu(),
            "speaker_embedding": speaker_embedding.detach().cpu(),
            "model": "XTTS-v2",
            "language": "es"
        }, profile_path)
        generated_profiles.append(profile_path)
        print(f"✓ {voice_id}: {profile_path.name}")
    except Exception as exc:
        failed_profiles.append((voice_id, str(exc)))
        print(f"✗ {voice_id}: {exc}")

print(f"\nPerfiles generados: {len(generated_profiles)}/{len(voice_sample_paths)}")
if failed_profiles:
    print("Revisar:", [voice_id for voice_id, _ in failed_profiles])

✓ ana: ana.pt
✓ diego: diego.pt
✓ elsa: elsa.pt
✓ estuardo: estuardo.pt
✓ nicole: nicole.pt
✓ winz: winz.pt

Perfiles generados: 6/6


## Perfiles creados

Los archivos `data/voice_profiles/<VOICE_ID>.pt` contienen las representaciones necesarias para sintetizar con XTTS. Restringe el acceso a las grabaciones y perfiles.

In [100]:
print("Perfiles disponibles para el asistente:")
for profile_path in sorted(VOICE_PROFILES_DIR.glob("*.pt")):
    print(" -", profile_path.stem, "→", profile_path.name)

Perfiles disponibles para el asistente:
 - ana → ana.pt
 - diego → diego.pt
 - elsa → elsa.pt
 - estuardo → estuardo.pt
 - nicole → nicole.pt
 - winz → winz.pt


## Probar síntesis

Selecciona un perfil y genera audio con un texto nuevo.

In [101]:
# Aqui selector de voz/perfil
available_profiles = sorted(p.stem for p in VOICE_PROFILES_DIR.glob("*.pt"))
if not available_profiles:
    raise FileNotFoundError(f"No hay perfiles .pt en {VOICE_PROFILES_DIR}")
profile_selector = widgets.Dropdown(
    options=available_profiles,
    value=available_profiles[0],
    description="Perfil de voz:",
    style={"description_width": "initial"}
)
display(profile_selector)
VOICE_ID_PRUEBA = profile_selector.value
print("Perfil seleccionado para la prueba:", VOICE_ID_PRUEBA)

Dropdown(description='Perfil de voz:', options=('ana', 'diego', 'elsa', 'estuardo', 'nicole', 'winz'), style=D…

Perfil seleccionado para la prueba: ana


In [102]:
# Leer el valor actual del menú al ejecutar esta celda.
VOICE_ID_PRUEBA = profile_selector.value
print("Generando audio con el perfil:", VOICE_ID_PRUEBA)
profile_path = VOICE_PROFILES_DIR / f"{VOICE_ID_PRUEBA}.pt"
profile = torch.load(profile_path, map_location=DEVICE, weights_only=True)
text = "Hola, quiero realizar compras en línea y necesito ayuda para encontrar los mejores productos."
with torch.inference_mode():
    result = model.inference(
        text=text, language="es",
        gpt_cond_latent=profile["gpt_cond_latent"].to(DEVICE),
        speaker_embedding=profile["speaker_embedding"].to(DEVICE),
        enable_text_splitting=True
    )
from IPython.display import Audio, display
display(Audio(result["wav"], rate=24000))

Generando audio con el perfil: ana


## Batería de pruebas de perfiles y síntesis

Revisa cinco puntos del flujo: lectura de WAV, perfiles guardados, selección del perfil, carga de sus representaciones y generación de audio. La tabla compara lo esperado con lo que existe en esta ejecución.

In [103]:
import json as _json
import pandas as pd
from IPython.display import display as _display

def resultado_prueba_voz(nombre, esperado, real, ok):
    return {"Prueba": nombre, "Esperado": esperado, "Resultado real": real,
            "Estado": "OK" if ok else "REVISAR"}

perfiles_prueba = sorted(p.stem for p in VOICE_PROFILES_DIR.glob("*.pt"))
wav_ids_prueba = sorted(p.stem for p in VOICE_SAMPLES_DIR.glob("*.wav"))
filas_pruebas_voz = []
filas_pruebas_voz.append(resultado_prueba_voz(
    "1. Grabaciones disponibles", "Hay al menos un WAV de referencia",
    f"{len(wav_ids_prueba)} WAV: {wav_ids_prueba}", bool(wav_ids_prueba)))
filas_pruebas_voz.append(resultado_prueba_voz(
    "2. Perfiles guardados", "Cada WAV tiene su archivo .pt",
    f"{len(perfiles_prueba)} perfiles: {perfiles_prueba}",
    bool(wav_ids_prueba) and set(wav_ids_prueba).issubset(set(perfiles_prueba))))
perfil_seleccionado_actual = profile_selector.value if "profile_selector" in globals() else None
filas_pruebas_voz.append(resultado_prueba_voz(
    "3. Selección de perfil", "El perfil seleccionado existe en disco",
    str(perfil_seleccionado_actual), perfil_seleccionado_actual in perfiles_prueba))

perfil_cargado = None
if perfil_seleccionado_actual in perfiles_prueba:
    perfil_cargado = torch.load(VOICE_PROFILES_DIR / f"{perfil_seleccionado_actual}.pt",
                               map_location=DEVICE, weights_only=True)
campos_requeridos = {"gpt_cond_latent", "speaker_embedding"}
campos_presentes = campos_requeridos.issubset(perfil_cargado or {})
filas_pruebas_voz.append(resultado_prueba_voz(
    "4. Representaciones de voz", "El perfil incluye latentes y speaker embedding",
    str(sorted((perfil_cargado or {}).keys())), campos_presentes))

audio_generado = globals().get("result")
audio_valido = audio_generado is not None and len(audio_generado.get("wav", [])) > 0
filas_pruebas_voz.append(resultado_prueba_voz(
    "5. Síntesis de audio", "La celda de prueba generó una forma de onda",
    f"audio generado={audio_valido}", audio_valido))

tabla_pruebas_voz = pd.DataFrame(filas_pruebas_voz)
_display(tabla_pruebas_voz)

,Prueba,Esperado,Resultado real,Estado
0,1. Grabaciones disponibles,Hay al menos un WAV de referencia,"6 WAV: ['ana', 'diego', 'elsa', 'estuardo', 'n...",OK
1,2. Perfiles guardados,Cada WAV tiene su archivo .pt,"6 perfiles: ['ana', 'diego', 'elsa', 'estuardo...",OK
2,3. Selección de perfil,El perfil seleccionado existe en disco,ana,OK
3,4. Representaciones de voz,El perfil incluye latentes y speaker embedding,"['gpt_cond_latent', 'language', 'model', 'spea...",OK
4,5. Síntesis de audio,La celda de prueba generó una forma de onda,audio generado=True,OK
